# DeepWalk: Online Learning of Social Representations

**Implementation from scratch** — uniform random walks + Skip-gram with negative sampling, compared against node2vec's biased walks.

**Paper:** Bryan Perozzi, Rami Al-Rfou, Steven Skiena. *DeepWalk: Online Learning of Social Representations*. KDD 2014. [arXiv:1403.6652](https://arxiv.org/abs/1403.6652)

DeepWalk learns node embeddings by treating truncated random walks on a graph as "sentences" and applying the Skip-gram model (from word2vec). Nodes that appear near each other in random walks tend to share communities, so the learned embeddings encode graph structure in a low-dimensional space.

## 1. Setup & Imports

In [1]:
# Install required packages (Colab/Kaggle compatible)
import subprocess, sys
for pkg in ['networkx', 'scikit-learn']:
    try:
        __import__(pkg if pkg != 'scikit-learn' else 'sklearn')
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import numpy as np
import networkx as nx
from collections import defaultdict, Counter
import random
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score as nmi_score
from sklearn.metrics import silhouette_score

np.random.seed(42)
random.seed(42)
print("Imports ready.")
print(f"NumPy: {np.__version__}, NetworkX: {nx.__version__}")

Imports ready.
NumPy: 2.1.3, NetworkX: 3.6.1


## 2. Graph Construction

We build a synthetic graph with **3 communities** (each a clustered subgraph) connected by bridge edges. This mirrors the structure of real social networks: tight communities with sparse inter-community links.

**Deliberate simplification:** The paper uses BlogCatalog (10,312 nodes), Flickr, and YouTube (1,138,499 nodes). We use a small synthetic graph so the full pipeline runs in under 5 minutes and embeddings can be visualized in 2D.

In [2]:
def build_synthetic_graph():
    """Build a synthetic graph with 3 communities and bridge edges."""
    G = nx.Graph()
    communities = {}
    node_idx = 0

    # 3 communities of ~25 nodes each
    for comm_id in range(3):
        comm_nodes = list(range(node_idx, node_idx + 25))
        for n in comm_nodes:
            communities[n] = comm_id
        G.add_nodes_from(comm_nodes)

        # Dense intra-community edges (each node connects to ~6 random others)
        for n in comm_nodes:
            neighbors = random.sample([x for x in comm_nodes if x != n], min(6, 24))
            for nb in neighbors:
                G.add_edge(n, nb)
        node_idx += 25

    # Bridge edges between communities (sparse)
    bridges = [(5, 30), (10, 55), (20, 40), (45, 70), (15, 65), (25, 50)]
    for u, v in bridges:
        G.add_edge(u, v)

    return G, communities

G, communities = build_synthetic_graph()
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Communities: {len(set(communities.values()))}")
print(f"Community sizes: {Counter(communities.values())}")

# Visualize the graph
plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G, seed=42)
colors = [communities[n] for n in G.nodes()]
nx.draw(G, pos, node_color=colors, cmap=plt.cm.Set1, node_size=80,
        with_labels=False, edge_color='gray', alpha=0.7, width=0.5)
plt.title("Synthetic Social Graph - 3 Communities", fontsize=14)
plt.tight_layout()
plt.savefig('graph_structure.png', dpi=100, bbox_inches='tight')
plt.show()
print("Graph visualized.")

Graph: 75 nodes, 399 edges
Communities: 3
Community sizes: Counter({0: 25, 1: 25, 2: 25})


Graph visualized.


/tmp/ipykernel_24/1512579571.py:40: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


## 3. Uniform Random Walk Generation (DeepWalk Core)

DeepWalk generates **truncated random walks** on the graph. Starting from each node, the walk moves to a uniformly-random neighbor at each step. Walks are treated as "sentences" where nodes are "words."

**Key property:** DeepWalk walks are **uniform** — every neighbor is equally likely. No bias parameters. (node2vec generalizes this with biased second-order walks using parameters p and q.)

In [3]:
def deepwalk_random_walk(G, start_node, walk_length):
    """Generate a single truncated random walk from start_node."""
    walk = [start_node]
    current = start_node
    for _ in range(walk_length - 1):
        neighbors = list(G.neighbors(current))
        if not neighbors:
            break
        current = random.choice(neighbors)
        walk.append(current)
    return walk

def generate_deepwalk_walks(G, num_walks, walk_length):
    """Generate random walks for every node in the graph."""
    walks = []
    nodes = list(G.nodes())
    for _ in range(num_walks):
        random.shuffle(nodes)
        for node in nodes:
            walk = deepwalk_random_walk(G, node, walk_length)
            walks.append(walk)
    return walks

# Generate DeepWalk walks
NUM_WALKS = 10
WALK_LENGTH = 20
walks_dw = generate_deepwalk_walks(G, NUM_WALKS, WALK_LENGTH)
print(f"Generated {len(walks_dw)} DeepWalk walks of length {WALK_LENGTH}")
print(f"Sample walk (first 10 nodes): {walks_dw[0][:10]}")
print(f"Total walk tokens: {sum(len(w) for w in walks_dw)}")

Generated 750 DeepWalk walks of length 20
Sample walk (first 10 nodes): [26, 41, 26, 36, 48, 40, 49, 43, 39, 43]
Total walk tokens: 15000


## 4. Skip-gram with Negative Sampling (From Scratch)

We implement Skip-gram with negative sampling from scratch using NumPy. For each node (center) in a walk, we predict the nodes within a context window. Positive pairs are updated to increase their dot product; negative samples are drawn from a unigram^0.75 distribution and pushed toward zero.

**Deliberate simplification:** The paper uses Hierarchical Softmax for scalability to millions of nodes. We use negative sampling (simpler, equally effective on small graphs).

In [4]:
def build_vocab(walks):
    """Map node IDs to integer indices and build frequency table."""
    node_set = set()
    for walk in walks:
        node_set.update(walk)
    node_list = sorted(node_set)
    vocab = {node: idx for idx, node in enumerate(node_list)}
    freq = Counter()
    for walk in walks:
        freq.update(walk)
    freq_array = np.array([freq[node] for node in node_list], dtype=np.float64)
    neg_freq = freq_array ** 0.75
    neg_freq /= neg_freq.sum()
    return vocab, node_list, freq_array, neg_freq

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -10, 10)))

def skipgram_train(walks, vocab_size, dim=16, window_size=3,
                   neg_samples=5, epochs=5, lr=0.025):
    """Train Skip-gram with negative sampling from scratch."""
    W_in = (np.random.uniform(-0.5, 0.5, (vocab_size, dim)) / dim).astype(np.float64)
    W_out = (np.random.uniform(-0.5, 0.5, (vocab_size, dim)) / dim).astype(np.float64)

    neg_table_size = 100000
    neg_table = np.random.choice(vocab_size, size=neg_table_size, p=neg_freq)
    neg_ptr = 0

    total_pairs = 0
    for walk in walks:
        for i in range(len(walk)):
            for j in range(max(0, i - window_size), min(len(walk), i + window_size + 1)):
                if i != j:
                    total_pairs += 1

    print(f"Training Skip-gram: {vocab_size} nodes, dim={dim}, {epochs} epochs, ~{total_pairs} pairs/epoch")

    for epoch in range(epochs):
        epoch_loss = 0.0
        pairs = 0
        cur_lr = lr * (1 - epoch / epochs)
        cur_lr = max(cur_lr, lr * 0.0001)

        for walk in walks:
            for i in range(len(walk)):
                center = walk[i]
                for j in range(max(0, i - window_size), min(len(walk), i + window_size + 1)):
                    if i == j:
                        continue
                    context = walk[j]

                    # Positive pair
                    score = np.dot(W_in[center], W_out[context])
                    sig = sigmoid(score)
                    grad = 1.0 - sig

                    grad_center = grad * W_out[context]
                    grad_context = grad * W_in[center]
                    epoch_loss += -np.log(sig + 1e-10)

                    # Negative samples
                    for _ in range(neg_samples):
                        neg_idx = neg_table[neg_ptr]
                        neg_ptr = (neg_ptr + 1) % neg_table_size
                        if neg_idx == context:
                            continue
                        neg_score = np.dot(W_in[center], W_out[neg_idx])
                        neg_sig = sigmoid(neg_score)
                        neg_grad = -neg_sig

                        grad_center += neg_grad * W_out[neg_idx]
                        W_out[neg_idx] += cur_lr * neg_grad * W_in[center]
                        epoch_loss += -np.log(1 - neg_sig + 1e-10)

                    W_in[center] += cur_lr * grad_center
                    W_out[context] += cur_lr * grad_context
                    pairs += 1

        print(f"  Epoch {epoch+1}/{epochs} - loss: {epoch_loss/max(pairs,1):.4f} - lr: {cur_lr:.5f}")

    return W_in

# Build vocab and train DeepWalk embeddings
vocab, node_list, freq_array, neg_freq = build_vocab(walks_dw)
print(f"Vocab size: {len(vocab)}")

walks_dw_idx = [[vocab[n] for n in walk] for walk in walks_dw]

embeddings_dw = skipgram_train(
    walks_dw_idx, len(vocab), dim=16, window_size=3,
    neg_samples=5, epochs=5, lr=0.025
)
print(f"\nDeepWalk embeddings shape: {embeddings_dw.shape}")

Vocab size: 75
Training Skip-gram: 75 nodes, dim=16, 5 epochs, ~81000 pairs/epoch


  Epoch 1/5 - loss: 2.1239 - lr: 0.02500


  Epoch 2/5 - loss: 1.7672 - lr: 0.02000


  Epoch 3/5 - loss: 1.7552 - lr: 0.01500


  Epoch 4/5 - loss: 1.7523 - lr: 0.01000


  Epoch 5/5 - loss: 1.7505 - lr: 0.00500

DeepWalk embeddings shape: (75, 16)


## 5. Biased Random Walks (node2vec, for Comparison)

To compare DeepWalk against node2vec, we implement biased second-order random walks. node2vec adds parameters:
- **p (return parameter):** likelihood of returning to the previous node (high p -> BFS-like)
- **q (in-out parameter):** likelihood of moving away from the previous node (low q -> DFS-like)

When p=q=1, node2vec reduces to DeepWalk's uniform walk. We test two settings:
- **BFS-like (p=1, q=2):** captures structural equivalence (nodes with similar roles)
- **DFS-like (p=1, q=0.5):** captures homophily (nodes in the same community)

In [5]:
def alias_setup(probs):
    """Construct alias sampling tables for O(1) sampling."""
    n = len(probs)
    probs = np.array(probs, dtype=np.float64)
    probs = probs * n / probs.sum()
    small = []
    large = []
    for i, p in enumerate(probs):
        if p < 1.0:
            small.append(i)
        else:
            large.append(i)

    alias = [0] * n
    prob = [0] * n
    while small and large:
        l = small.pop()
        g = large.pop()
        prob[l] = probs[l]
        alias[l] = g
        probs[g] = probs[g] + probs[l] - 1.0
        if probs[g] < 1.0:
            small.append(g)
        else:
            large.append(g)
    while large:
        g = large.pop()
        prob[g] = 1.0
    while small:
        l = small.pop()
        prob[l] = 1.0
    return prob, alias

def alias_draw(prob, alias):
    """O(1) draw from alias table."""
    n = len(prob)
    i = random.randint(0, n - 1)
    if random.random() < prob[i]:
        return i
    return alias[i]

def compute_node2vec_transition_probs(G, p, q):
    """Precompute 2nd-order transition probabilities for biased random walks."""
    alias_tables = {}
    for curr in G.nodes():
        neighbors = list(G.neighbors(curr))
        if not neighbors:
            continue
        alias_tables[curr] = {}
        for prev in neighbors + [curr]:
            weights = []
            for nxt in neighbors:
                if nxt == prev:
                    w = 1.0 / p
                elif G.has_edge(prev, nxt):
                    w = 1.0
                else:
                    w = 1.0 / q
                weights.append(w)
            alias_tables[curr][prev] = alias_setup(weights)
    return alias_tables

def biased_random_walk(G, start, walk_length, p, q, alias_tables):
    """Simulate a single biased second-order random walk."""
    walk = [start]
    current = start
    prev = start
    for step in range(walk_length - 1):
        neighbors = list(G.neighbors(current))
        if not neighbors:
            break
        if step == 0:
            nxt = random.choice(neighbors)
        else:
            if current in alias_tables and prev in alias_tables[current]:
                prob, alias = alias_tables[current][prev]
                idx = alias_draw(prob, alias)
                nxt = neighbors[idx]
            else:
                nxt = random.choice(neighbors)
        walk.append(nxt)
        prev = current
        current = nxt
    return walk

def generate_node2vec_walks(G, num_walks, walk_length, p, q):
    """Generate biased random walks for all nodes."""
    alias_tables = compute_node2vec_transition_probs(G, p, q)
    walks = []
    nodes = list(G.nodes())
    for _ in range(num_walks):
        random.shuffle(nodes)
        for node in nodes:
            walk = biased_random_walk(G, node, walk_length, p, q, alias_tables)
            walks.append(walk)
    return walks

# Generate node2vec walks with two parameter settings
walks_bfs = generate_node2vec_walks(G, NUM_WALKS, WALK_LENGTH, p=1, q=2.0)
walks_dfs = generate_node2vec_walks(G, NUM_WALKS, WALK_LENGTH, p=1, q=0.5)

print(f"BFS-like walks (p=1, q=2): {len(walks_bfs)} walks")
print(f"DFS-like walks (p=1, q=0.5): {len(walks_dfs)} walks")

walks_bfs_idx = [[vocab[n] for n in walk] for walk in walks_bfs]
walks_dfs_idx = [[vocab[n] for n in walk] for walk in walks_dfs]

print("\nTraining BFS-like node2vec embeddings...")
embeddings_bfs = skipgram_train(
    walks_bfs_idx, len(vocab), dim=16, window_size=3,
    neg_samples=5, epochs=5, lr=0.025
)

print("\nTraining DFS-like node2vec embeddings...")
embeddings_dfs = skipgram_train(
    walks_dfs_idx, len(vocab), dim=16, window_size=3,
    neg_samples=5, epochs=5, lr=0.025
)
print(f"\nBFS embeddings shape: {embeddings_bfs.shape}")
print(f"DFS embeddings shape: {embeddings_dfs.shape}")

BFS-like walks (p=1, q=2): 750 walks
DFS-like walks (p=1, q=0.5): 750 walks

Training BFS-like node2vec embeddings...
Training Skip-gram: 75 nodes, dim=16, 5 epochs, ~81000 pairs/epoch


  Epoch 1/5 - loss: 2.1008 - lr: 0.02500


  Epoch 2/5 - loss: 1.6916 - lr: 0.02000


  Epoch 3/5 - loss: 1.6825 - lr: 0.01500


  Epoch 4/5 - loss: 1.6748 - lr: 0.01000


...

  Epoch 1/5 - loss: 2.1494 - lr: 0.02500


  Epoch 2/5 - loss: 1.7966 - lr: 0.02000


  Epoch 3/5 - loss: 1.7913 - lr: 0.01500


  Epoch 4/5 - loss: 1.7857 - lr: 0.01000


  Epoch 5/5 - loss: 1.7871 - lr: 0.00500

BFS embeddings shape: (75, 16)
DFS embeddings shape: (75, 16)


## 6. Evaluation & Comparison

We evaluate embedding quality using:
- **t-SNE visualization:** 2D projection of 16-dim embeddings, colored by ground-truth community
- **K-Means clustering:** cluster the embeddings, compare to ground-truth communities
- **NMI (Normalized Mutual Information):** quantitative measure of community recovery (1.0 = perfect)
- **Silhouette Score:** cluster cohesion/separation in embedding space

We compare: DeepWalk (uniform) vs node2vec BFS-like (p=1, q=2) vs node2vec DFS-like (p=1, q=0.5).

In [6]:
def evaluate_embeddings(embeddings, communities, node_list):
    """Evaluate embedding quality: t-SNE viz, K-Means, NMI, silhouette."""
    gt_labels = np.array([communities[node] for node in node_list])

    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(node_list)-1))
    emb_2d = tsne.fit_transform(embeddings)

    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(embeddings)

    nmi = nmi_score(gt_labels, cluster_labels)

    if len(set(cluster_labels)) > 1:
        sil = silhouette_score(embeddings, cluster_labels)
    else:
        sil = 0.0

    return emb_2d, cluster_labels, gt_labels, nmi, sil

# Evaluate all three methods
results = {}
methods = {
    'DeepWalk (uniform)': embeddings_dw,
    'node2vec BFS (p=1, q=2)': embeddings_bfs,
    'node2vec DFS (p=1, q=0.5)': embeddings_dfs,
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cmap = plt.cm.Set1

for ax, (name, emb) in zip(axes, methods.items()):
    emb_2d, cluster_labels, gt_labels, nmi, sil = evaluate_embeddings(
        emb, communities, node_list
    )
    results[name] = {'nmi': nmi, 'silhouette': sil, 'cluster_labels': cluster_labels}

    scatter = ax.scatter(emb_2d[:, 0], emb_2d[:, 1], c=gt_labels, cmap=cmap,
                         s=50, alpha=0.8, edgecolors='black', linewidths=0.3)
    ax.set_title(f"{name}\nNMI={nmi:.3f}, Silhouette={sil:.3f}", fontsize=11)
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')

plt.suptitle('Node Embeddings: DeepWalk vs node2vec (colored by ground-truth community)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('embedding_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n" + "=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
for name, r in results.items():
    print(f"  {name:30s} - NMI: {r['nmi']:.4f}, Silhouette: {r['silhouette']:.4f}")
print("=" * 60)


EVALUATION RESULTS
  DeepWalk (uniform)             - NMI: 1.0000, Silhouette: 0.4922
  node2vec BFS (p=1, q=2)        - NMI: 1.0000, Silhouette: 0.4439
  node2vec DFS (p=1, q=0.5)      - NMI: 1.0000, Silhouette: 0.5425


## 7. Graph Visualization with Learned Clusters

Visualize the original graph with nodes colored by K-Means clusters from each method's embeddings. This shows how well each walk strategy recovers the community structure.

In [7]:
def graph_with_clusters(G, embeddings, k=3, ax=None, title=''):
    """Draw graph with nodes colored by K-Means clusters on embeddings."""
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(embeddings)

    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 6))

    nx.draw(G, pos, node_color=cluster_labels, cmap=plt.cm.Set1,
            node_size=80, with_labels=False, edge_color='gray',
            alpha=0.7, width=0.5, ax=ax)
    ax.set_title(title, fontsize=12)
    return cluster_labels

fig, axes = plt.subplots(1, 4, figsize=(24, 5))

# Ground truth
gt_labels = [communities[n] for n in G.nodes()]
nx.draw(G, pos, node_color=gt_labels, cmap=plt.cm.Set1, node_size=80,
        with_labels=False, edge_color='gray', alpha=0.7, width=0.5, ax=axes[0])
axes[0].set_title('Ground Truth Communities', fontsize=12)

# DeepWalk
graph_with_clusters(G, embeddings_dw, ax=axes[1], title='DeepWalk (uniform)')

# node2vec BFS
graph_with_clusters(G, embeddings_bfs, ax=axes[2], title='node2vec BFS (p=1, q=2)')

# node2vec DFS
graph_with_clusters(G, embeddings_dfs, ax=axes[3], title='node2vec DFS (p=1, q=0.5)')

plt.tight_layout()
plt.savefig('graph_clusters_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("Graph cluster visualization complete.")

Graph cluster visualization complete.


## 8. Summary & Results

### What we implemented

1. **Uniform random walks** — DeepWalk's core: truncated walks where each neighbor is equally likely
2. **Skip-gram with negative sampling** — from scratch in NumPy, with unigram^0.75 negative sampling and linear LR decay
3. **Biased second-order random walks** — node2vec's generalization with alias sampling for O(1) draws
4. **Comparison framework** — t-SNE visualization, K-Means clustering, NMI, and silhouette scores

### Key takeaways

- **DeepWalk = node2vec with p=q=1.** DeepWalk's uniform walks are the simplest case of node2vec's biased walks.
- On **homophily-dominated** graphs (tight communities), DeepWalk and DFS-like node2vec perform similarly — both recover community structure well.
- **BFS-like** node2vec (q=2) captures structural equivalence differently — it may group nodes by role (e.g., hubs) rather than by community.
- DeepWalk's advantage is **simplicity and online learning** — no hyperparameter tuning for walk bias, and new walks can be added incrementally.
- node2vec's advantage is **flexibility** — the p and q parameters let it adapt to graphs where structural equivalence matters as much as homophily.

In [8]:
print("=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)
print()
print("DeepWalk vs node2vec: Community Recovery (NMI)")
print("-" * 50)
for name, r in results.items():
    bar_len = int(r['nmi'] * 30)
    print(f"  {name:30s} NMI: {r['nmi']:.4f} [{'#' * bar_len}{'.' * (30 - bar_len)}]")
    print(f"  {'':30s} Silhouette: {r['silhouette']:.4f}")
    print()

best_method = max(results, key=lambda x: results[x]['nmi'])
print(f"Best NMI: {best_method} ({results[best_method]['nmi']:.4f})")
print()
print("DeepWalk (uniform walks) is the foundation of random-walk-based")
print("graph embeddings. node2vec generalizes it with biased walks, but")
print("DeepWalk remains competitive on homophily-dominated graphs.")
print("=" * 60)

FINAL RESULTS SUMMARY

DeepWalk vs node2vec: Community Recovery (NMI)
--------------------------------------------------
  DeepWalk (uniform)             NMI: 1.0000 [##############################]
                                 Silhouette: 0.4922

  node2vec BFS (p=1, q=2)        NMI: 1.0000 [##############################]
                                 Silhouette: 0.4439

  node2vec DFS (p=1, q=0.5)      NMI: 1.0000 [##############################]
                                 Silhouette: 0.5425

Best NMI: DeepWalk (uniform) (1.0000)

DeepWalk (uniform walks) is the foundation of random-walk-based
graph embeddings. node2vec generalizes it with biased walks, but
DeepWalk remains competitive on homophily-dominated graphs.
